The MMASH module predicts stress status using only five HRV-based features computed over 5-minute sliding windows. A Multi-Layer Perceptron (MLP) Neural Network Classifier for the MMASH Dataset is trained with strict user-level data separation to ensure the model generalizes to entirely new, previously unseen individuals — a critical requirement for real-world stress monitoring tools.

Standalone Production Predict & Full Evaluation Function

In [2]:
# ==============================================================================
# Standalone Cloud Production Predict & Evaluation Engine
# File: MultiLayerPerceptronMLP_MMASH_Pred.ipynb
# ==============================================================================
import os
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report,
    matthews_corrcoef, cohen_kappa_score, log_loss, brier_score_loss
)

# 1. Mount Google Drive safely (skips remount prompt if already mounted)
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    else:
        print("[INFO] Google Drive is already mounted.")
except ImportError:
    print("[INFO] Running in non-Colab environment.")

# 2. Self-Contained Directory & Path Declarations
BASE_DIR = '/content/drive/MyDrive/MMASH_Model'
INPUT_DIR = os.path.join(BASE_DIR, 'InputData')
OUTPUT_MODEL_DIR = os.path.join(BASE_DIR, 'OutPutModel')
OUTPUT_FILES_DIR = os.path.join(BASE_DIR, 'OutPutFiles')

# Ensure output directory exists
os.makedirs(OUTPUT_FILES_DIR, exist_ok=True)

# Explicit File Paths
INPUT_CSV_PATH = os.path.join(INPUT_DIR, 'MMASH_NSRI_Preprocessed_Final.csv')
FINAL_MODEL_PATH = os.path.join(OUTPUT_MODEL_DIR, 'MultiLayerPerceptron_MLP_MMASH_Model01.pkl')
PRED_RESULTS_PATH = os.path.join(OUTPUT_FILES_DIR, 'MultiLayerPerceptron_MLP_MMASH_Predt01.csv')
FULL_METRICS_PATH = os.path.join(OUTPUT_FILES_DIR, 'MultiLayerPerceptron_MLP_MMASH_FullPred_Metrics.csv')
FULL_REPORT_PATH = os.path.join(OUTPUT_FILES_DIR, 'MultiLayerPerceptron_MLP_MMASH_FullPred_ClassificationReport.csv')


# 3. Main Standalone Predict & Comprehensive Evaluation Engine
def predict_stress_mlp(
    input_csv_path,
    model_path,
    output_pred_csv,
    output_metrics_csv=None,
    output_report_csv=None
):
    """
    Self-contained standalone inference and evaluation engine using Multi-Layer Perceptron:
    - Loads pre-trained MLP Pipeline (SimpleImputer + StandardScaler + MLPClassifier).
    - Dynamically maps 5 HRV physiological features regardless of casing.
    - Applies physiological rule-based stress labelling if ground truth is absent.
    - Computes continuous stress probabilities and confidence percentages.
    - Evaluates 15 comprehensive performance metrics & classification reports.
    - Exports all tabular results to Drive.
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Trained model artifact not found at: {model_path}")
    if not os.path.exists(input_csv_path):
        raise FileNotFoundError(f"Input dataset not found at: {input_csv_path}")

    # Load trained model pipeline and input dataset
    model_pipeline = joblib.load(model_path)
    data = pd.read_csv(input_csv_path)
    print(f"[INFERENCE] Loaded {len(data)} rows from '{os.path.basename(input_csv_path)}'")

    # Extract and match 5 HRV features dynamically
    HRV_FEATURES = ['HR', 'SDNN', 'RMSSD', 'pNN50', 'LF_HF']
    feature_cols = []
    for feat in HRV_FEATURES:
        match = next((col for col in data.columns if col.lower() == feat.lower()), None)
        if match:
            feature_cols.append(match)
        else:
            alt_match = next((col for col in data.columns if feat.lower() in col.lower()), None)
            if alt_match:
                feature_cols.append(alt_match)

    if len(feature_cols) == 0:
        raise ValueError("Could not find required HRV feature columns in input dataset.")

    print(f"[FEATURES DETECTED] Using columns: {feature_cols}")
    X_in = data[feature_cols].copy()

    # Model Inference
    preds = model_pipeline.predict(X_in)
    probas = model_pipeline.predict_proba(X_in)
    stress_prob = probas[:, 1] if probas.shape[1] > 1 else probas[:, 0]
    confidence = np.max(probas, axis=1) * 100.0

    # Assemble Output DataFrame
    output_df = data.copy()
    output_df['Predicted_Target'] = preds
    output_df['Predicted_Label'] = output_df['Predicted_Target'].map({0: 'Non-Stress', 1: 'Stress'})
    output_df['Stress_Probability'] = np.round(stress_prob, 6)
    output_df['Prediction_Confidence'] = np.round(confidence, 2)
    output_df['Model_Used'] = 'MultiLayerPerceptron_MLP (MMASH)'

    # Ground Truth Label Verification or Generation
    TARGET_COL = 'stress_target'
    if TARGET_COL not in output_df.columns or output_df[TARGET_COL].isnull().all():
        print("\n[GROUND TRUTH] 'stress_target' missing in raw file. Applying rule-based labelling...")
        hr_c = next((c for c in feature_cols if 'hr' in c.lower() and 'lf' not in c.lower()), feature_cols[0])
        sdnn_c = next((c for c in feature_cols if 'sdnn' in c.lower()), feature_cols[1])
        rmssd_c = next((c for c in feature_cols if 'rmssd' in c.lower()), feature_cols[2])

        p75_hr = output_df[hr_c].quantile(0.75)
        p25_sdnn = output_df[sdnn_c].quantile(0.25)
        p25_rmssd = output_df[rmssd_c].quantile(0.25)

        output_df[TARGET_COL] = np.where(
            (output_df[hr_c] > p75_hr) & ((output_df[sdnn_c] < p25_sdnn) | (output_df[rmssd_c] < p25_rmssd)),
            1,
            0
        )

    # 15 Comprehensive Metric Suite Evaluation
    y_true = output_df[TARGET_COL].astype(int)
    y_pred = output_df['Predicted_Target'].astype(int)

    output_df['Actual_Label'] = y_true.map({0: 'Non-Stress', 1: 'Stress'})
    output_df['Prediction_Status'] = np.where(y_true == y_pred, 'Correct', 'Incorrect')

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metrics_dict = {
        'Metric': [
            'Accuracy',
            'Balanced Accuracy',
            'Precision (Stress)',
            'Recall / Sensitivity',
            'Specificity',
            'F1-Score (Weighted)',
            'F1-Score (Macro)',
            'ROC-AUC',
            'Matthews Corr Coef (MCC)',
            'Cohen Kappa',
            'Log Loss',
            'Brier Score',
            'Total Windows Evaluated',
            'Correct Predictions',
            'Incorrect Predictions',
            'Mean Prediction Confidence (%)'
        ],
        'Score': [
            accuracy_score(y_true, y_pred),
            balanced_accuracy_score(y_true, y_pred),
            precision_score(y_true, y_pred, zero_division=0),
            recall_score(y_true, y_pred, zero_division=0),
            specificity,
            f1_score(y_true, y_pred, average='weighted', zero_division=0),
            f1_score(y_true, y_pred, average='macro', zero_division=0),
            roc_auc_score(y_true, stress_prob),
            matthews_corrcoef(y_true, y_pred),
            cohen_kappa_score(y_true, y_pred),
            log_loss(y_true, stress_prob),
            brier_score_loss(y_true, stress_prob),
            len(y_true),
            int((y_true == y_pred).sum()),
            int((y_true != y_pred).sum()),
            np.mean(confidence)
        ]
    }
    metrics_df = pd.DataFrame(metrics_dict)

    # Classification Report
    clf_dict = classification_report(
        y_true, y_pred,
        target_names=['Non-Stress (0)', 'Stress (1)'],
        output_dict=True
    )
    clf_report_df = pd.DataFrame(clf_dict).transpose()

    print("\n" + "=" * 68)
    print("      FULL MMASH MULTILAYER PERCEPTRON EVALUATION METRICS")
    print("=" * 68)
    display(metrics_df.round(4))

    print("\n" + "=" * 68)
    print("            FULL PREDICTION CLASSIFICATION REPORT")
    print("=" * 68)
    display(clf_report_df.round(4))

    # Save Summaries to OutputFiles
    if output_metrics_csv:
        metrics_df.to_csv(output_metrics_csv, index=False)
        print(f"[SAVED] Metrics Summary -> {output_metrics_csv}")

    if output_report_csv:
        clf_report_df.to_csv(output_report_csv)
        print(f"[SAVED] Classification Report -> {output_report_csv}")

    # Save Detailed Row Predictions
    output_df.to_csv(output_pred_csv, index=False)
    print(f"[SUCCESS] Prediction report for {len(output_df)} samples exported to: {output_pred_csv}")

    return output_df, metrics_df


# 4. Execute Standalone Prediction Pipeline
pred_df, metrics_summary = predict_stress_mlp(
    input_csv_path=INPUT_CSV_PATH,
    model_path=FINAL_MODEL_PATH,
    output_pred_csv=PRED_RESULTS_PATH,
    output_metrics_csv=FULL_METRICS_PATH,
    output_report_csv=FULL_REPORT_PATH
)

# 5. Preview Output
preview_candidates = [
    col for col in pred_df.columns
    if any(k in col.lower() for k in ['user', 'subject', 'hr', 'sdnn', 'rmssd'])
]
display_cols = list(dict.fromkeys(preview_candidates + [
    'stress_target', 'Predicted_Target', 'Stress_Probability', 'Prediction_Confidence', 'Prediction_Status'
]))
display_cols = [c for c in display_cols if c in pred_df.columns]

print("\n--- Preview of Generated Predictions ---")
display(pred_df[display_cols].head(10))

[INFO] Google Drive is already mounted.
[INFERENCE] Loaded 5863 rows from 'MMASH_NSRI_Preprocessed_Final.csv'
[FEATURES DETECTED] Using columns: ['mean_hr', 'sdnn', 'rmssd']

[GROUND TRUTH] 'stress_target' missing in raw file. Applying rule-based labelling...

      FULL MMASH MULTILAYER PERCEPTRON EVALUATION METRICS


,Metric,Score
0,Accuracy,0.9794
1,Balanced Accuracy,0.9730
2,Precision (Stress),0.8947
3,Recall / Sensitivity,0.9642
4,Specificity,0.9818
5,F1-Score (Weighted),0.9797
6,F1-Score (Macro),0.9581
7,ROC-AUC,0.9968
8,Matthews Corr Coef (MCC),0.9170
9,Cohen Kappa,0.9162



            FULL PREDICTION CLASSIFICATION REPORT


,precision,recall,f1-score,support
Non-Stress (0),0.9942,0.9818,0.9879,5052.0000
Stress (1),0.8947,0.9642,0.9282,811.0000
accuracy,0.9794,0.9794,0.9794,0.9794
macro avg,0.9445,0.9730,0.9581,5863.0000
weighted avg,0.9804,0.9794,0.9797,5863.0000


[SAVED] Metrics Summary -> /content/drive/MyDrive/MMASH_Model/OutPutFiles/MultiLayerPerceptron_MLP_MMASH_FullPred_Metrics.csv
[SAVED] Classification Report -> /content/drive/MyDrive/MMASH_Model/OutPutFiles/MultiLayerPerceptron_MLP_MMASH_FullPred_ClassificationReport.csv
[SUCCESS] Prediction report for 5863 samples exported to: /content/drive/MyDrive/MMASH_Model/OutPutFiles/MultiLayerPerceptron_MLP_MMASH_Predt01.csv

--- Preview of Generated Predictions ---


,user_id,mean_hr,sdnn,rmssd,mean_hr_z,sdnn_z,rmssd_z,stress_target,Predicted_Target,Stress_Probability,Prediction_Confidence,Prediction_Status
0,user_1,96.290542,98.863267,70.058724,1.403664,0.647901,0.651154,0,0,0.000010,100.00,Correct
1,user_1,89.385475,57.303510,34.783170,0.945479,-0.587178,-0.482878,0,1,0.671058,67.11,Incorrect
2,user_1,83.322651,62.037952,31.367692,0.543181,-0.446479,-0.592678,0,0,0.142742,85.73,Correct
3,user_1,92.740481,114.396096,44.421161,1.168100,1.109508,-0.173038,0,0,0.000755,99.92,Correct
4,user_1,83.487582,77.645612,47.365125,0.554125,0.017352,-0.078396,0,0,0.000979,99.90,Correct
5,user_1,84.712706,65.826318,39.407586,0.635418,-0.333896,-0.334213,0,0,0.037529,96.25,Correct
6,user_1,93.469395,76.115032,33.218822,1.216467,-0.028134,-0.533169,0,0,0.374574,62.54,Correct
7,user_1,105.206674,60.665992,24.084643,1.995292,-0.487251,-0.826812,1,1,0.992887,99.29,Correct
8,user_1,94.537278,66.235062,34.263327,1.287326,-0.321748,-0.499590,0,0,0.479929,52.01,Correct
9,user_1,90.067418,65.557051,34.705504,0.990729,-0.341898,-0.485375,0,0,0.366695,63.33,Correct
